# ANIMORA — Phase 1: Data Engineering, Preprocessing & EDA

**ANIMORA** is a portfolio-grade, full-stack anime recommendation engine.
This notebook documents and visualizes the core Phase 1 milestones:
1. **Data Ingestion**: Reproducible acquisition of verified MyAnimeList datasets.
2. **Data Cleaning & Sanitization**: Deduplication, sentinel replacement, text cleaning, genre normalization.
3. **Feature Engineering**: Bayesian weighted ratings, log-transformed popularity, content soup for NLP.
4. **Exploratory Data Analysis (EDA)**: Statistical distributions, popularity behavior, format analysis, and correlation insights.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from ml.config import (
    RAW_METADATA_PATH,
    PROCESSED_CSV_PATH,
    PROCESSED_PARQUET_PATH,
    FEATURE_METADATA_PATH,
    FIGURES_DIR,
)
from ml.data_loader import load_raw_datasets
from ml.preprocessor import clean_anime_dataset
from ml.feature_engineering import build_and_save_pipeline
from ml.eda import run_eda_pipeline

print(f"Project root resolved to: {BASE_DIR}")

## 1. Data Ingestion & Raw Overview

We ingest over 17,500 anime records across 3 authoritative MyAnimeList raw datasets:
- `anime.csv`: Comprehensive metadata (title, scores, format, episodes, dates, studio, members, ranking)
- `anime_with_synopsis.csv`: Rich textual summaries for NLP feature representation
- `myanimelist.csv`: Community metrics and CDN image poster links

In [ ]:
df_meta, df_syn, df_mal = load_raw_datasets()
print(f"Raw Metadata: {df_meta.shape}")
print(f"Raw Synopsis: {df_syn.shape}")
if df_mal is not None:
    print(f"Raw Extra MAL: {df_mal.shape}")
df_meta[["MAL_ID", "Name", "Score", "Type", "Episodes", "Members"]].head()

## 2. Preprocessing & Feature Engineering Execution

Running the end-to-end cleaning and feature engineering pipeline:
- Replaces sentinel missing values (`Unknown`, `?`, `-`)
- Filters unusable records missing titles or genre labels
- Deduplicates records by MAL ID and normalized title
- Computes IMDB/MAL style Bayesian Weighted Ratings ($WR$)
- Calculates $\log(1 + x)$ transformations on community member counts
- Assembles rich `content_soup` token strings for downstream TF-IDF and transformer embeddings

In [ ]:
df_clean, meta = build_and_save_pipeline()
print(f"Successfully processed {len(df_clean):,} anime titles with {len(df_clean.columns)} features!")
df_clean[["mal_id", "name", "score", "weighted_score", "type", "release_year", "members", "primary_genre"]].head(10)

### Inspecting Engineered Content Soup
The `content_soup` combines title, alternate titles, format, studio, source, genres, and sanitized synopsis for content-based models.

In [ ]:
sample = df_clean.iloc[0]
print(f"Title: {sample['name']}")
print(f"Genres: {sample['genres']}")
print(f"Weighted Score: {sample['weighted_score']}")
print(f"\nContent Soup (first 350 chars):\n{sample['content_soup'][:350]}...")

## 3. Exploratory Data Analysis (Visualizations & Insights)
Visualizing distributions, correlations, and domain insights.

In [ ]:
# Display generated EDA figures
display(Image(filename=str(FIGURES_DIR / "02_rating_distribution.png")))
display(Image(filename=str(FIGURES_DIR / "03_popularity_distribution.png")))
display(Image(filename=str(FIGURES_DIR / "04_genre_frequency.png")))
display(Image(filename=str(FIGURES_DIR / "05_type_distribution.png")))
display(Image(filename=str(FIGURES_DIR / "06_episode_distribution.png")))
display(Image(filename=str(FIGURES_DIR / "07_release_year_trend.png")))
display(Image(filename=str(FIGURES_DIR / "08_correlation_matrix.png")))
display(Image(filename=str(FIGURES_DIR / "09_score_vs_popularity.png")))

## 4. Phase 1 Key Findings Summary

1. **Data Integrity**: 17,495 usable records retained out of 17,562 (99.6% retention rate).
2. **Rating Dynamics**: Global average score is **6.51**. Raw ratings exhibit high variance for low-vote titles; Bayesian regularization smoothly dampens unvoted outliers.
3. **Popularity Skew**: Raw member counts have massive skew (median 1,080 vs max 2,589,507). The log-transformed metric `log_members` restores Gaussian symmetry.
4. **Genre Landscape**: **Comedy** (6,029), **Action** (3,887), **Fantasy** (3,297), and **Adventure** (2,957) dominate the anime medium.
5. **Broadcast Structure**: TV anime strictly cluster around standard Japanese television cours (12 episodes for 1-cour, 24–26 episodes for 2-cours).
6. **Industry Production Trend**: Anime production peaked during 2016–2018 with ~800+ titles annually before transitioning to higher-budget streaming titles.